# Reference Data Standardization

This notebook walks through how the reference data used throughout the xQTL protocol is downloaded, formatted, and indexed. Each step calls a single workflow from the `reference_data_preparation.ipynb` module (plus `VCF_QC.ipynb` for the dbSNP step).

**Timing:** ~4 hours, dominated by the genome download and the STAR/RSEM indexing steps.

## Description

The workflow downloads the raw reference files, standardizes the genome and gene annotations, builds the STAR and RSEM alignment indices, and generates the RefFlat, SUPPA, and dbSNP annotations used downstream:

1. Download the reference genome, gene annotation, ERCC spike-in reference, and dbSNP variants.
2. Format the reference genome and gene annotation (merge ERCC, standardize the FASTA/GTF).
3. Build the STAR and RSEM indices.
4. Generate the RefFlat (Picard), SUPPA (Psichomics), and dbSNP rsID annotations.

## Input Files

The download steps need only internet access; nothing local is required to begin. The later formatting and indexing steps consume the products of the preceding steps under the `output/reference_data/` working directory.

| File | Description |
| :--- | :--- |
| `GRCh38_full_analysis_set_plus_decoy_hla.fa` | Reference genome FASTA (downloaded) |
| `Homo_sapiens.GRCh38.103.chr.gtf` | Ensembl gene annotation GTF (downloaded) |
| `ERCC92.fa` / `ERCC92.gtf` | ERCC spike-in reference (downloaded) |
| `00-All.vcf.gz` | dbSNP variant reference (downloaded) |

## Steps

### 1. [Download Reference Data](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data_preparation.html#i-download-reference-data)

Timing: ~2 hours

In [ ]:
sos run pipeline/reference_data_preparation.ipynb download_hg_reference --cwd output/reference_data
sos run pipeline/reference_data_preparation.ipynb download_gene_annotation --cwd output/reference_data
sos run pipeline/reference_data_preparation.ipynb download_ercc_reference --cwd output/reference_data
sos run pipeline/reference_data_preparation.ipynb download_dbsnp --cwd output/reference_data

### 2. [Format Reference Data](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data_preparation.html#ii-format-reference-data)

Timing: ~40 min


In [ ]:
sos run pipeline/reference_data_preparation.ipynb hg_reference \
    --cwd output/reference_data \
    --ercc-reference output/reference_data/ERCC92.fa \
    --hg-reference output/reference_data/GRCh38_full_analysis_set_plus_decoy_hla.fa 

Change the --stranded option to --no-stranded if using un-stranded RNA-seq protocol. 

In [ ]:
sos run pipeline/reference_data_preparation.ipynb hg_gtf \
    --cwd output/reference_data \
    --hg-gtf output/reference_data/Homo_sapiens.GRCh38.103.chr.gtf \
    --hg-reference output/reference_data/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy.fasta \
    --stranded 

### 3. [Format Gene Feature Data](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data_preparation.html#iii-format-gene-feature-data)

Timing: ~1min

Change the --stranded option to --no-stranded if using un-stranded RNA-seq protocol. 

In [ ]:
sos run pipeline/reference_data_preparation.ipynb gene_annotation \
    --cwd output/reference_data \
    --ercc-gtf output/reference_data/ERCC92.gtf \
    --hg-gtf output/reference_data/Homo_sapiens.GRCh38.103.chr.gtf \
    --hg-reference output/reference_data/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy.fasta \
    --stranded 

### 4. [Generate STAR Index](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data_preparation.html#iv-generate-star-index)

Timing: ~30 min

Generate the STAR index without the GTF annotation file to allow for customized read length later in STAR alignment. 

In [ ]:
sos run pipeline/reference_data_preparation.ipynb STAR_index \
    --cwd output/reference_data \
    --hg-reference output/reference_data/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy_ERCC.fasta \
    --numThreads 10 \
    --mem 40G

### 5. [Generate RSEM Index](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data_preparation.html#v-generate-rsem-index)

Timing: ~1min

Generate RSEM index with the uncollapsed gtf file (without the gene tag in its file name.)


In [ ]:
sos run pipeline/reference_data_preparation.ipynb RSEM_index \
    --cwd output/reference_data \
    --hg-reference output/reference_data/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy_ERCC.fasta \
    --hg-gtf output/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.gtf

### 6. [Generate RefFlat Annotation for Picard](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data_preparation.html#vi-generate-refflat-annotation-for-picard)

Timing: ~1min

In [ ]:
sos run pipeline/reference_data_preparation.ipynb RefFlat_generation \
    --cwd output/reference_data \
    --hg-gtf output/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.gtf

### 7. [Generate SUPPA Annotation for Psichomics](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data_preparation.html#vii-generate-suppa-annotation-for-psichomics)

Timing: ~1min

In [ ]:
sos run pipeline/reference_data_preparation.ipynb SUPPA_annotation \
    --cwd output/reference_data \
    --hg_gtf output/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.gtf 

### 8. [Extract rsIDs for Known Variants](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data_preparation.html#viii-extract-rsids-for-known-variants)

Timing: <30min

Extract the rsID for the known varriants so that we can distinguish them from the novel variants we had in our study. The procedure/rationale is explained in this post

In [ ]:
sos run pipeline/VCF_QC.ipynb dbsnp_annotate \
    --genoFile output/reference_data/00-All.vcf.gz \
    --cwd output/reference_data 

## Output

The steps above write a standardized reference bundle that the rest of the protocol consumes:

| File | Produced by |
| --- | --- |
| Cleaned reference genome (FASTA) and matched gene model (GTF) | Steps 1–2 (download & format reference data) |
| Per-feature gene-coordinate table | Step 3 (format gene feature data) |
| STAR genome index | Step 4 (generate STAR index) |
| RSEM reference | Step 5 (generate RSEM index) |
| RefFlat annotation (for Picard RNA-seq metrics) | Step 6 |
| SUPPA event annotation (for psichomics splicing) | Step 7 |
| rsID lookup for known variants | Step 8 (extract rsIDs) |

## Anticipated Results

Running all eight steps produces a complete, standardized reference-data bundle for the rest of the protocol: a cleaned reference genome and matched gene-model annotation (GTF), per-feature gene-coordinate tables, a STAR genome index and an RSEM reference for RNA-seq alignment/quantification, a RefFlat annotation for Picard RNA-seq metrics, a SUPPA event annotation for psichomics-based splicing analysis, and an rsID lookup for known variants. These files are written once and then reused as fixed inputs by the molecular-phenotype, association, and downstream modules, so that every analysis in the protocol is run against the same curated reference.